# Assembly data structure generation

This notebook builds a single per-assembly table for the tumor cohort. **Each row is one assembly.**

Columns:
- `subject_id` - anonymized subject (from `subj_list`)
- `insertion_index` - overall insertion index (0..27), same indexing as the neuron table
- `assembly_index` - which assembly within that insertion (0-based, ICA order)
- `region` - from `region_list`
- `flair` - 1 if `opercular_list == 0`, else 0
- `grade` - from `grade_list`
- `path` - from `path_list`
- `weight_vector` - the normalized assembly pattern `ap_norm_matrix[assembly_index, :]`, one weight per good neuron in the insertion
- `beh_tstat_prod` - production-activity t-statistic (NaN if the session has no speech production)
- `beh_tstat_rec` - reception-activity t-statistic (NaN if the session has no speech reception)
- `information_capacity` - assembly information capacity (computed from the ICA component time series)
- `expression_strength` - raw expression-strength time series across the recording

In [4]:
import numpy as np
import pandas as pd
import pickle
import fnmatch
import re
from pathlib import Path

import pynapple as nap
from scipy import stats
from scipy.stats import kstest, ttest_rel
from scipy.linalg import norm
from sklearn.decomposition import PCA, FastICA

# Canonical tumor-cohort session list (22 NWB files, 28 insertions).
# Every descriptive list below is indexed by INSERTION (0..27).
nwb_paths = [
    Path("/data_store2/neuropixels/nwb/old/NP93_B1/NP93_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP95_B1/NP95_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP101_B3/NP101_B3.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP105_B1/NP105_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP113_B1/NP113_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP114_B1/NP114_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP116_B2/NP116_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP122_B1/NP122_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP128_B1/NP128_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP129_B1/NP129_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP132_B2/NP132_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP132_B3/NP132_B3.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP136_B1/NP136_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP137_B1/NP137_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP138_B1/NP138_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP139_B1/NP139_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP139_B2/NP139_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP147_B2/NP147_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP150_B1/NP150_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP153_B1/NP153_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP171_B1/NP171_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP174_B3/NP174_B3.nwb"),
]

# Per-insertion descriptive info (length 28).
subj_list      = [1, 2, 3, 5, 6, 6, 6, 7, 8, 9, 10, 10, 11, 12, 13, 14, 15, 16, 16, 17, 17, 18, 19, 20, 20, 21, 22, 23]
path_list      = ['ast', 'ast', 'gbm', 'oli', 'gbm', 'gbm', 'gbm', 'gbm', 'gbm', 'ast', 'ast', 'ast', 'ast', 'oli', 'oli', 'gbm', 'ast', 'ast', 'ast', 'ast', 'ast', 'ast', 'gbm', 'ast', 'ast', 'oli', 'gbm', 'gbm']
grade_list     = [4, 4, 4, 3, 4, 4, 4, 4, 4, 3, 2, 2, 4, 2, 2, 4, 2, 2, 2, 2, 2, 2, 4, 2, 2, 3, 4, 4]
opercular_list = [1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1]
region_list    = ['aSTG', 'SFG', 'aSTG', 'SFG', 'vPrCG', 'vPrCG', 'vPrCG', 'pSTG', 'aMTG', 'MFG', 'aSTG', 'parsOp', 'MFG', 'PoCG', 'PoCG', 'pSTG', 'parsOr', 'vPrCG', 'pSTG', 'parsTr', 'pSTG', 'parsTr', 'SMG', 'parsTr', 'pSTG', 'vPrCG', 'aSTG', 'vPrCG']

manual_exclude_lists = [
    [2],                                                                                                                                                                              # NP93_B1.imec0
    [],                                                                                                                                                                               # NP95_B1.imec0
    [29, 45, 55],                                                                                                                                                                     # NP101_B3.imec0
    [130, 150, 151, 245, 371, 375, 377, 380, 386, 452],                                                                                                                              # NP105_B1.imec0
    [149, 156, 172, 173, 176, 217, 221, 253, 272, 276, 336, 487],                                                                                                                    # NP113_B1.imec0
    [22, 112, 149, 155, 161, 167, 176, 188, 190, 196, 213, 216, 241, 252, 255, 258, 286, 289, 307, 325, 326, 370, 395, 442, 443, 448, 449, 451, 455, 510, 549, 582, 692, 710, 717, 719, 722, 726, 729],  # NP113_B1.imec1
    [385, 448],                                                                                                                                                                       # NP113_B1.imec2
    [32, 34, 37, 39, 67, 142, 145, 175, 188, 310, 325, 331, 391, 392],                                                                                                               # NP114_B1.imec0
    [11, 15, 30, 36],                                                                                                                                                                 # NP116_B2.imec0
    [340, 382],                                                                                                                                                                       # NP122_B1.imec0
    [54, 109, 152],                                                                                                                                                                   # NP128_B1.imec0
    [],                                                                                                                                                                               # NP128_B1.imec1
    [0, 6, 13, 33, 51, 52],                                                                                                                                                           # NP129_B1.imec0
    [19, 62, 111, 151, 171, 192, 199, 200, 205, 210, 227, 229, 244, 290, 298, 311],                                                                                                  # NP132_B2.imec0
    [0, 10, 11, 15, 29, 45, 61, 112, 127, 149, 151, 158, 159, 172, 197, 209, 212, 218, 297, 315],                                                                                    # NP132_B3.imec0
    [334, 335, 333],                                                                                                                                                                  # NP136_B1.imec0
    [361],                                                                                                                                                                            # NP137_B1.imec0
    [],                                                                                                                                                                               # NP138_B1.imec0
    [169, 195, 198],                                                                                                                                                                  # NP138_B1.imec1
    [137, 149, 196, 197, 199, 206, 212, 220, 223, 231],                                                                                                                              # NP139_B1.imec0
    [129, 248, 249],                                                                                                                                                                  # NP139_B1.imec1
    [28, 59, 60, 62, 63, 82, 104, 112, 146],                                                                                                                                          # NP139_B2.imec0
    [47, 57, 59, 73, 82, 88, 116, 124, 125, 126, 127, 129, 148, 175, 194, 196, 239, 240, 242, 243, 244, 246, 253, 267, 245, 247, 248, 249, 250, 251, 254],                          # NP147_B2.imec0
    [],                                                                                                                                                                               # NP150_B1.imec0
    [34, 64, 119, 135],                                                                                                                                                               # NP150_B1.imec1
    [20, 22, 47, 71, 80, 118, 125, 134, 141, 145, 146, 166, 173, 176, 191, 270, 298, 307, 385, 414, 391, 472, 473, 490, 491],                                                        # NP153_B1.imec0
    [16, 67, 74, 78, 91, 99, 221, 224, 225, 288],                                                                                                                                     # NP171_B1.imec0
    [9, 11, 13, 23, 40, 43, 47, 59, 67, 72, 87, 104, 109, 122, 166, 169, 172, 175, 197, 199, 203, 205, 211, 212, 219, 227, 229, 230, 235, 237, 240, 251, 256, 295, 296],            # NP174_B3.imec0
]

assert len(subj_list) == len(path_list) == len(grade_list) == len(opercular_list) == len(region_list) == len(manual_exclude_lists) == 28
print("NWB files:", len(nwb_paths))
print("insertions:", len(subj_list))

NWB files: 22
insertions: 28


In [5]:
# Main loop: identify assemblies per insertion (same process as R_F3_assemblybehavior /
# R_F3_SF5_assemblyinfo) and assemble one row per assembly.

def imec_key_sorter(key):
    m = re.search(r"imec(\d+)", key)
    return int(m.group(1)) if m else float("inf")

MIN_BEHAVIOR_MINUTES = 0   # matches the reference notebooks (always uses the TaskTimes window)
TIMESCALE = 25 / 1000      # 25 ms bins
PROD_REL_START, PROD_REL_END = -0.10, 0.0
SENS_REL_START, SENS_REL_END = 0.0, 0.10

rows = []
insertion = 0

for i in range(len(nwb_paths)):
    data = nap.load_file(nwb_paths[i])
    keys = data.keys()

    keys = [k for k in keys if fnmatch.fnmatch(k, "*imec*")]
    ks_keys = [k for k in keys if "KS4" in k]
    if ks_keys:
        keys = ks_keys
    th8_keys = [k for k in keys if "Th=8" in k]
    if th8_keys:
        keys = th8_keys
    else:
        th_keys = [k for k in keys if "Th=" in k]
        if th_keys:
            keys = th_keys
    keys = [k for k in keys if not fnmatch.fnmatch(k, "*sentgen*") and not fnmatch.fnmatch(k, "*_auto*")]
    if ("NP137" in str(nwb_paths[i])) or ("NP139_B2" in str(nwb_paths[i])):
        keys = [k for k in keys if "imec1" not in k]
    keys = sorted(keys, key=imec_key_sorter)

    for s in range(len(keys)):
        spike_times = data[keys[s]]
        firingRates_all = spike_times.metadata["rate"]

        if "TaskTimes" in data.keys():
            task_times = data["TaskTimes"]
        else:
            task_times = data["task_times"]
        beh_epochs = nap.IntervalSet(start=task_times.start, end=task_times.end)

        starts = np.asarray(task_times.start, dtype=float).ravel()
        ends = np.asarray(task_times.end, dtype=float).ravel()
        beh_sec = float(np.sum(ends - starts)) if starts.size == ends.size else np.nan
        beh_min = beh_sec / 60.0 if np.isfinite(beh_sec) else np.nan
        use_full_recording = (not np.isfinite(beh_min)) or (beh_min < MIN_BEHAVIOR_MINUTES)

        if use_full_recording:
            firingRates_beh = firingRates_all
            tmin, tmax = np.inf, -np.inf
            for u in range(len(spike_times)):
                idx = spike_times[u].as_series().index.values
                if len(idx):
                    tmin = min(tmin, float(np.min(idx)))
                    tmax = max(tmax, float(np.max(idx)))
            if np.isfinite(tmin) and np.isfinite(tmax):
                min_time, max_time = tmin, tmax
            else:
                min_time, max_time = np.nan, np.nan
        else:
            spike_times_beh = spike_times.restrict(beh_epochs)
            firingRates_beh = spike_times_beh.metadata["rate"]
            min_time = starts[0]
            max_time = ends[-1]

        # KS statistic vs uniform over [min_time, max_time]
        ks_stats = np.zeros(len(spike_times))
        for u in range(len(spike_times)):
            t = spike_times[u].as_series().index.values
            if len(t) > 1:
                ks_stats[u] = kstest((t - min_time) / (max_time - min_time), "uniform").statistic
            else:
                ks_stats[u] = np.nan

        # ISI refractory violation percentage (3 ms)
        violationPct = np.zeros(len(spike_times))
        for u in range(len(spike_times)):
            unit = spike_times[u].as_series().index
            if len(unit) < 100:
                violationPct[u] = 1
            else:
                isi = unit.diff()[1:]
                violationPct[u] = np.array(np.where(isi < 3 / 1000)).size / len(isi)

        if "KSLabel" in spike_times.metadata:
            KSLabels = spike_times.metadata["KSLabel"]
        else:
            KSLabels = spike_times.metadata["quality"]

        firingRates = firingRates_beh
        mask = (violationPct < 3 / 100) & (firingRates > 0.5) & (KSLabels != "noise") & (ks_stats < 0.3)
        indicesFinal = firingRates.index[mask]
        indicesFinal = np.setdiff1d(indicesFinal, manual_exclude_lists[insertion])

        # ---- per-insertion descriptive info ----
        subj = subj_list[insertion]
        region = region_list[insertion]
        grade = grade_list[insertion]
        path = path_list[insertion]
        flair = 1 if opercular_list[insertion] == 0 else 0

        # ---- 25 ms binned, z-scored firing-rate matrix (full recording) ----
        spike_times_good = spike_times[indicesFinal]
        spikeCountMatrix = spike_times_good.count(bin_size=TIMESCALE)
        bin_edges = spikeCountMatrix.index.values
        bin_centers = bin_edges[:-1] + np.diff(bin_edges) / 2
        if len(bin_centers) < len(bin_edges):
            last_width = bin_edges[-1] - bin_edges[-2] if len(bin_edges) > 1 else 0
            last_center = bin_edges[-1] - last_width / 2
            bin_centers = np.append(bin_centers, last_center)
        firingRateMatrix = spikeCountMatrix.values / TIMESCALE
        firingRateMatrix = stats.zscore(firingRateMatrix, axis=0)

        # ---- PCA -> number of assemblies (Marchenko-Pastur upper bound) ----
        n_assemblies = 0
        if firingRateMatrix.shape[0] > 1 and firingRateMatrix.shape[1] > 0:
            pca = PCA()
            pca.fit(firingRateMatrix)
            eigenvalues = pca.explained_variance_
            upperbound = (1 + np.sqrt(firingRateMatrix.shape[1] / firingRateMatrix.shape[0])) ** 2
            n_assemblies = int(np.sum(eigenvalues > upperbound))

        print(f"insertion {insertion:2d}  {nwb_paths[i].stem:12s} {keys[s]:38s}  subj {subj:2d}  {region:7s}  neurons {len(indicesFinal):3d}  assemblies {n_assemblies}")

        if n_assemblies == 0:
            insertion += 1
            continue

        # ---- FastICA -> assembly patterns (verbatim from the reference) ----
        pc_vectors = pca.components_[:n_assemblies, :]
        projections = firingRateMatrix @ pc_vectors.T
        fastica = FastICA(
            n_components=n_assemblies,
            algorithm="parallel",
            whiten="unit-variance",
            max_iter=500,
            tol=1e-7,
            random_state=1,
        )
        ica_components = fastica.fit_transform(projections)        # shape (time_bins, n_assemblies)
        unmixing_matrix = fastica.components_
        ica_assembly_patterns = unmixing_matrix @ pc_vectors        # shape (n_assemblies, n_neurons)

        ap_norm_matrix = ica_assembly_patterns.copy()
        for ii in range(n_assemblies):
            ap = ica_assembly_patterns[ii, :]
            ap_norm = ap / norm(ap)
            if abs(np.min(ap_norm)) > np.max(ap_norm):
                ap_norm = ap_norm * -1
            ap_norm_matrix[ii, :] = ap_norm

        # ---- expression strength time series for each assembly ----
        # E(b) = R(b)^T (w w^T) R(b) == (R(b) . w)^2  (vectorized; matches reference to fp precision)
        expression_strength_matrix = (firingRateMatrix @ ap_norm_matrix.T) ** 2   # (time_bins, n_assemblies)
        significance_thresholds = np.mean(expression_strength_matrix, axis=0)      # per-assembly mean

        bin_starts = bin_centers - TIMESCALE / 2
        bin_ends = bin_centers + TIMESCALE / 2

        # ---- production / reception t-stats (R_F3_assemblybehavior) ----
        prod_tstat = np.full(n_assemblies, np.nan)
        rec_tstat = np.full(n_assemblies, np.nan)

        if "ProdSpeechWords" in data:
            sp = data["ProdSpeechWords"]
            prod_starts = nap.Ts(sp.start).as_series().index + PROD_REL_START
            prod_ends = nap.Ts(sp.end).as_series().index + PROD_REL_END
            for a in range(n_assemblies):
                assembly_expression = expression_strength_matrix[:, a]
                prod_expression = np.zeros(len(prod_starts))
                for u in range(len(prod_starts)):
                    in_interval = (bin_starts >= prod_starts[u]) & (bin_ends <= prod_ends[u])
                    prod_expression[u] = np.mean(assembly_expression[in_interval]) if np.any(in_interval) else np.nan
                if len(prod_expression) > 1:
                    tval, _ = ttest_rel(prod_expression, np.full_like(prod_expression, significance_thresholds[a]), alternative="greater")
                else:
                    tval = np.nan
                prod_tstat[a] = tval

        if ("StimSpeechWords" in data) or ("mfa_stim_words" in data):
            sp = data["StimSpeechWords"] if "StimSpeechWords" in data else data["mfa_stim_words"]
            sens_starts = nap.Ts(sp.start).as_series().index + SENS_REL_START
            sens_ends = nap.Ts(sp.end).as_series().index + SENS_REL_END
            for a in range(n_assemblies):
                assembly_expression = expression_strength_matrix[:, a]
                sens_expression = np.zeros(len(sens_starts))
                for u in range(len(sens_starts)):
                    in_interval = (bin_starts >= sens_starts[u]) & (bin_ends <= sens_ends[u])
                    sens_expression[u] = np.mean(assembly_expression[in_interval]) if np.any(in_interval) else np.nan
                if len(sens_expression) > 1:
                    tval, _ = ttest_rel(sens_expression, np.full_like(sens_expression, significance_thresholds[a]), alternative="greater")
                else:
                    tval = np.nan
                rec_tstat[a] = tval

        # ---- assembly information capacity (R_F3_SF5_assemblyinfo) ----
        # Computed from each assembly's ICA component time series. capacity = entropy - temporal MI.
        info_cap = np.full(n_assemblies, np.nan)
        for a in range(n_assemblies):
            comp = ica_components[:, a]
            valid = comp[~np.isnan(comp)]
            if len(valid) < 10:
                capacity = np.nan
            else:
                n_bins = min(20, len(valid) // 5)
                if n_bins < 5:
                    n_bins = 5
                hist, _ = np.histogram(valid, bins=n_bins)
                prob = hist / np.sum(hist)
                prob = prob[prob > 0]
                entropy_val = -np.sum(prob * np.log2(prob)) if len(prob) >= 2 else np.nan

                x = valid[:-1]
                y = valid[1:]
                mi_bins = min(10, len(valid) // 10)
                vm = ~(np.isnan(x) | np.isnan(y))
                if np.sum(vm) < 2:
                    mi_temporal = np.nan
                else:
                    hist_2d, _, _ = np.histogram2d(x[vm], y[vm], bins=mi_bins)
                    total = np.sum(hist_2d)
                    p_xy = hist_2d / total
                    p_x = np.sum(hist_2d, axis=1) / total
                    p_y = np.sum(hist_2d, axis=0) / total
                    mi_temporal = 0.0
                    for aa in range(len(p_x)):
                        for bb in range(len(p_y)):
                            if p_xy[aa, bb] > 0 and p_x[aa] > 0 and p_y[bb] > 0:
                                mi_temporal += p_xy[aa, bb] * np.log2(p_xy[aa, bb] / (p_x[aa] * p_y[bb]))

                if not np.isnan(entropy_val) and not np.isnan(mi_temporal):
                    capacity = entropy_val - mi_temporal
                elif not np.isnan(entropy_val):
                    capacity = entropy_val
                else:
                    capacity = np.nan
            info_cap[a] = capacity

        # ---- assemble one row per assembly ----
        for a in range(n_assemblies):
            rows.append({
                "subject_id": int(subj),
                "insertion_index": int(insertion),
                "assembly_index": int(a),
                "region": region,
                "flair": int(flair),
                "grade": int(grade),
                "path": path,
                "weight_vector": np.asarray(ap_norm_matrix[a, :], dtype=float),
                "beh_tstat_prod": float(prod_tstat[a]),
                "beh_tstat_rec": float(rec_tstat[a]),
                "information_capacity": info_cap[a],
                "expression_strength": np.asarray(expression_strength_matrix[:, a], dtype=float),
            })

        insertion += 1

assert insertion == len(subj_list), (insertion, len(subj_list))
print("\\ninsertions processed:", insertion)
print("total assemblies:", len(rows))

/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/base_class.py:60: UserWarning: Some epochs have no duration
  self.time_support = IntervalSet(start=self.index[0], end=self.index[-1])
/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/base_class.py:62: RuntimeWarning: divide by zero encountered in scalar divide
  self.rate = self.index.shape[0] / np.sum(
/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/metadata_class.py:188: UserWarning: Metadata name 'Task name' contains a special character, and cannot be accessed as an attribute. Use 'get_info()' or key indexing to access metadata.
  warnings.warn(


insertion  0  NP93_B1      NP93_B1_g0_imec0_withoutKSmc            subj  1  aSTG     neurons   1  assemblies 0


/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/metadata_class.py:188: UserWarning: Metadata name 'Task name' contains a special character, and cannot be accessed as an attribute. Use 'get_info()' or key indexing to access metadata.
  warnings.warn(


insertion  1  NP95_B1      NP95_B1_g0_imec1                        subj  2  SFG      neurons  57  assemblies 11


/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/io/interface_nwb.py:133: UserWarning: Some starts and ends are equal. Removing 1 microsecond!
  data = nap.IntervalSet(df)
/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/io/interface_nwb.py:133: UserWarning: Some starts and ends are equal. Removing 1 microsecond!
  data = nap.IntervalSet(df)
/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/metadata_class.py:188: UserWarning: Metadata name 'Task name' contains a special character, and cannot be accessed as an attribute. Use 'get_info()' or key indexing to access metadata.
  warnings.warn(


...

/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/io/interface_nwb.py:133: UserWarning: Some starts and ends are equal. Removing 1 microsecond!
  data = nap.IntervalSet(df)


insertion 26  NP171_B1     catgt_NP171_B1_g0_imec0_KS4_Th=8        subj 22  aSTG     neurons  17  assemblies 2


/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/core/metadata_class.py:188: UserWarning: Metadata name 'Task name' contains a special character, and cannot be accessed as an attribute. Use 'get_info()' or key indexing to access metadata.
  warnings.warn(


insertion 27  NP174_B3     catgt_NP174_B3_g0_imec0_KS4_Th=8        subj 23  vPrCG    neurons  24  assemblies 4
\ninsertions processed: 28
total assemblies: 154


/userdata/gumbach/miniforge3/envs/my_se2nwb/lib/python3.10/site-packages/pynapple/io/interface_nwb.py:133: UserWarning: Some starts and ends are equal. Removing 1 microsecond!
  data = nap.IntervalSet(df)


In [6]:
# Assemble the final per-assembly table and save it.
column_order = [
    "subject_id", "insertion_index", "assembly_index",
    "region", "flair", "grade", "path",
    "weight_vector",
    "beh_tstat_prod", "beh_tstat_rec", "information_capacity",
    "expression_strength",
]
assembly_data = pd.DataFrame(rows)[column_order]

print("shape:", assembly_data.shape)
print()
print("coverage:")
print("  beh_tstat_prod present:      ", int(assembly_data["beh_tstat_prod"].notna().sum()))
print("  beh_tstat_rec present:       ", int(assembly_data["beh_tstat_rec"].notna().sum()))
print("  information_capacity present:", int(assembly_data["information_capacity"].notna().sum()))
print()
print("assemblies per insertion:")
print(assembly_data.groupby("insertion_index").size())
print()
assembly_data.drop(columns=["weight_vector", "expression_strength"]).head(20)

shape: (154, 12)

coverage:
  beh_tstat_prod present:       143
  beh_tstat_rec present:        128
  information_capacity present: 154

assemblies per insertion:
insertion_index
1     11
2      3
4     18
5     23
6     22
7      9
8      1
9      5
10     1
11     5
13     6
14     9
15     4
16     6
17     2
19     5
20     2
21     5
22     5
23     1
24     2
25     3
26     2
27     4
dtype: int64



,subject_id,insertion_index,assembly_index,region,flair,grade,path,beh_tstat_prod,beh_tstat_rec,information_capacity
0,2,1,0,SFG,1,4,ast,2.649082,-1.534686,2.514961
1,2,1,1,SFG,1,4,ast,0.028933,0.745497,3.086005
2,2,1,2,SFG,1,4,ast,2.071547,0.545219,1.805345
3,2,1,3,SFG,1,4,ast,2.590470,-10.317335,2.263977
4,2,1,4,SFG,1,4,ast,-1.946387,3.697582,2.111673
5,2,1,5,SFG,1,4,ast,6.281920,-14.944574,2.621590
6,2,1,6,SFG,1,4,ast,1.023846,-0.546978,2.137910
7,2,1,7,SFG,1,4,ast,1.055383,1.173681,3.070253
8,2,1,8,SFG,1,4,ast,-2.149853,3.119708,2.407286
9,2,1,9,SFG,1,4,ast,1.509092,-1.980072,2.888157


In [7]:
with open("assembly_data_structure.pkl", "wb") as f:
    pickle.dump(assembly_data, f, protocol=pickle.HIGHEST_PROTOCOL)
print("saved assembly_data_structure.pkl  (rows:", len(assembly_data), ")")

saved assembly_data_structure.pkl  (rows: 154 )
